<a href="https://colab.research.google.com/github/samyakbaid/ML_coding_questions/blob/main/assingment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # 4-Factor Model: Market Stress and Model Stability (Indian Market, 2011-2022)
#
# This notebook implements the full methodological pipeline for the custom
# 4-factor model:
#
# $$R_{i,t} - R_{f,t} = \alpha_i + \beta_M (R_{M,t}-R_{f,t}) + \beta_L \Delta\text{VIX}_t + \beta_{FX}\, r^{INR}_t + \beta_{TS}\, \Delta\text{TermSpread}_t + \varepsilon_t$$
#
# Sectors: Bank, IT, Pharma, FMCG, Energy.  Frequency: weekly (Friday close).  Window: 2011-01-14 to 2022-12-30.
#
# **Pipeline stages**
#
# 1. **Build master panel** from raw uploads
# 2. **Diagnostics**       — verify OLS assumptions A1-A6 per (sector x regime)
# 3. **Inference**         — t / F / z tests + 95% CIs (HAC standard errors)
# 4. **Rolling estimation**— 52-week rolling betas + walk-forward OOS forecasts
# 5. **Regime-wise validity** — composite reliability score per regime
#
# **Required uploads** (4 CSV files):
#
# | Filename                                       | Source                                      |
# |------------------------------------------------|---------------------------------------------|
# | `tbill_91day_clean.csv`                        | RBI 91-day T-bill auction data              |
# | `nifty_weekly_2005_2022.csv`                   | output of the earlier `fetch_index_data.py` |
# | `macro_factors_weekly.csv`                     | output of `fetch_factors.py`                |
# | `India_10-Year_Bond_Yield_Historical_Data.csv` | investing.com export                        |
#
# Output files are written to a local `outputs/` directory and zipped at the end
# for download.

# %% [markdown]
# ## Cell 1 -- Install dependencies & set up paths

# %%
# In Colab, statsmodels and scipy are pre-installed. Just in case:
try:
    import statsmodels, scipy
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "statsmodels", "scipy", "pandas", "numpy"])

import os
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import (
    het_breuschpagan, het_white, het_arch,
    acorr_ljungbox, linear_reset,
)
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")

# Paths -- the input CSVs are expected in the current working directory.
# In Colab, run the next cell to upload the four required CSVs.
CWD     = Path.cwd()
INPUT   = CWD                          # uploaded CSVs land here in Colab
DATA    = CWD / "data"                 # intermediate CSVs
OUT     = CWD / "outputs"              # final result documents
TABLES  = OUT / "tables"
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)

print("Input directory :", INPUT)
print("Data directory  :", DATA)
print("Outputs root    :", OUT)
print("Tables folder   :", TABLES)

# %% [markdown]
# ## Cell 2 -- Upload the four required CSVs (Colab only)
#
# Run this cell in Google Colab to upload all four files at once.  If you are
# running locally, place the four CSVs in the same directory as the notebook
# and skip this cell.

# %%
try:
    from google.colab import files
    print("Colab detected. Use the file picker below to select the 4 CSVs.")
    uploaded = files.upload()
    for fname in uploaded:
        print(f"  uploaded: {fname}")
except ImportError:
    print("Not running in Colab. Make sure the 4 CSVs are in the working directory.")

# %% [markdown]
# ## Cell 3 -- Configuration constants

# %%
WINDOW_START = "2011-01-14"
WINDOW_END   = "2022-12-30"

FACTORS = ["mkt_excess", "d_vix", "r_inr", "d_term_spread"]
SECTORS = ["xr_BANK", "xr_IT", "xr_PHARMA", "xr_FMCG", "xr_ENERGY"]

# 10 calendar-defined regimes (4 crises + 5 calm + 1 mixed)
REGIMES = [
    ("Pre-Taper Calm",  "2011-01-14", "2013-04-26", "calm"),
    ("Taper Tantrum",   "2013-05-03", "2013-09-27", "crisis"),
    ("Mid-decade Calm", "2013-10-04", "2016-10-28", "calm"),
    ("Demonetization",  "2016-11-04", "2017-02-24", "crisis"),
    ("Pre-IL&FS Calm",  "2017-03-03", "2018-08-31", "calm"),
    ("IL&FS / NBFC",    "2018-09-07", "2019-03-29", "crisis"),
    ("Pre-COVID Calm",  "2019-04-05", "2020-01-31", "calm"),
    ("COVID Crash",     "2020-02-07", "2020-04-24", "crisis"),
    ("COVID Recovery",  "2020-05-01", "2021-12-31", "calm"),
    ("Tightening",      "2022-01-07", "2022-12-30", "mixed"),
]

# Crisis-vs-preceding-calm pairs for cross-regime tests
REGIME_PAIRS = [
    ("Pre-Taper Calm",  "Taper Tantrum"),
    ("Mid-decade Calm", "Demonetization"),
    ("Pre-IL&FS Calm",  "IL&FS / NBFC"),
    ("Pre-COVID Calm",  "COVID Crash"),
    ("COVID Crash",     "COVID Recovery"),
]

ROLLING_WINDOW = 52   # weeks
HAC_LAGS_FULL  = 6    # for regime-pooled inference
HAC_LAGS_ROLL  = 3    # Newey-West rule for T=52
ALPHA          = 0.05

COEF_MAP = {
    "const":         "alpha",
    "mkt_excess":    "beta_M",
    "d_vix":         "beta_L",
    "r_inr":         "beta_FX",
    "d_term_spread": "beta_TS",
}


def regime_type(name):
    for nm, _, _, t in REGIMES:
        if nm == name:
            return t
    return None


def assign_regime(d):
    for nm, s, e, _ in REGIMES:
        if pd.Timestamp(s) <= d <= pd.Timestamp(e):
            return nm
    return None


# %% [markdown]
# ## Cell 4 -- Methodology documentation
#
# Saved to `outputs/00_methodology.txt` so the report-ready text is on disk
# before any analytical output.

# %%
METHOD = r"""
================================================================================
4-FACTOR CUSTOM MODEL  --  METHODOLOGY DOCUMENT
================================================================================

MODEL SPECIFICATION
-------------------
For each sector i and week t:

    R_i,t  -  R_f,t  =  alpha_i
                       + beta_M  (R_M,t - R_f,t)            (market premium)
                       + beta_L  (Delta VIX_t)              (liquidity / risk-aversion)
                       + beta_FX (USDINR weekly return)     (FX / capital-flow channel)
                       + beta_TS (Delta term spread)        (monetary-policy stance)
                       + epsilon_t

estimated by ordinary least squares; inference uses Newey-West HAC standard
errors (heteroskedasticity-and-autocorrelation consistent).


VARIABLES
---------
R_M,t        : weekly Nifty 50 return (Friday-to-Friday simple return)
R_f,t        : weekly risk-free rate, derived from the 91-day T-bill annual
               yield via (1 + y/100)^(1/52) - 1
Delta VIX_t  : first difference of India VIX (level) week over week
r^INR_t      : USD/INR weekly percentage change (positive = rupee depreciates)
Delta TS_t   : first difference of (10Y G-Sec yield - 91-day T-bill yield)


OLS ASSUMPTIONS (A1-A6)
-----------------------
A1  Linearity            : E[y | X] is linear in factors.
                           Test: Ramsey RESET.
A2  No multicollinearity : columns of X are not perfectly collinear; ideally
                           pairwise correlations and VIFs are modest.
                           Test: VIF (rule: <5 ok, 5-10 watch, >10 problem).
A3  Exogeneity           : E[epsilon | X] = 0.  Not directly testable; argued
                           qualitatively (macro factors not caused by sector
                           returns at weekly frequency).
A4  Homoskedasticity     : Var(epsilon) constant across observations.
                           Tests: Breusch-Pagan, White, Engle ARCH-LM.
                           Failure expected for financial returns; HAC SEs fix it.
A5  No autocorrelation   : Cov(epsilon_t, epsilon_s) = 0 for t != s.
                           Tests: Durbin-Watson, Ljung-Box on residuals and
                           squared residuals.
A6  Normality of errors  : Tests: Jarque-Bera, plus skewness and excess kurtosis.
                           Failure expected for weekly equity returns; the
                           Central Limit Theorem rescues asymptotic inference
                           when n is sufficiently large (n >= 30 typical).


WHY HAC STANDARD ERRORS
-----------------------
Coefficient estimates from OLS are unbiased even when A4 (homoskedasticity) and
A5 (no autocorrelation) are violated.  However, classical OLS standard errors
become incorrect (typically too small), inflating t-statistics and producing
overconfident inference.  Newey-West HAC standard errors are computed using a
weighted long-run variance estimator that remains consistent under both
heteroskedasticity and autocorrelation.

In this study HAC standard errors are used for ALL inference: t-statistics,
p-values, 95% confidence intervals, F-tests for joint hypotheses, and Chow
tests for structural breaks across regimes.

Bandwidth choices:
    Regime-pooled inference   : maxlags = 6
    52-week rolling estimation: maxlags = 3  (Newey-West rule for T=52)


REGIMES
-------
Ten calendar-defined regimes covering the 2011-2022 window:

    Pre-Taper Calm   2011-01-14 to 2013-04-26   calm
    Taper Tantrum    2013-05-03 to 2013-09-27   crisis
    Mid-decade Calm  2013-10-04 to 2016-10-28   calm
    Demonetization   2016-11-04 to 2017-02-24   crisis
    Pre-IL&FS Calm   2017-03-03 to 2018-08-31   calm
    IL&FS / NBFC     2018-09-07 to 2019-03-29   crisis
    Pre-COVID Calm   2019-04-05 to 2020-01-31   calm
    COVID Crash      2020-02-07 to 2020-04-24   crisis
    COVID Recovery   2020-05-01 to 2021-12-31   calm
    Tightening       2022-01-07 to 2022-12-30   mixed


ROLLING ESTIMATION
------------------
Window length : 52 weeks (1-year)
Step          : 1 week
For each window ending at week t:
  - estimate the 4-factor model on weeks [t-51 .. t]
  - record alpha, beta_M, beta_L, beta_FX, beta_TS and HAC SEs at time t
  - forecast y_{t+1} from observed factors at t+1: y_pred = X_{t+1}' beta_hat
  - construct 95% prediction interval using sigma_hat^2 + X' Cov_HAC(beta) X


VALIDITY ASSESSMENT
-------------------
For each (sector x regime) cell, five components are computed and rescaled
to [0, 1]; their equally-weighted mean is the composite validity score.

  1. in-sample R^2          : from regime-pooled regression
  2. out-of-sample R^2      : 1 - SSE / sum(y^2), from rolling forecasts
  3. diagnostic pass rate   : fraction of {linearity, homoskedasticity,
                                            no-autocorrelation} tests passing
                              (normality is excluded -- failure is expected)
  4. PI coverage closeness  : 1 - |observed_coverage - 0.95|
  5. beta stability         : 1 / (1 + std(rolling beta_M within regime))


DELIVERABLES
------------
After running this notebook end-to-end, the outputs/ directory contains:

  outputs/00_methodology.txt              this document
  outputs/diag_summary.txt                readable diagnostics report
  outputs/inference_summary.txt           readable inference report
  outputs/rolling_summary.txt             readable rolling/OOS report
  outputs/validity_summary.txt            FINAL validity verdict per regime
  outputs/tables/diag_table.csv           60 cells x ~50 metrics
  outputs/tables/vif_by_regime.csv        VIF per regime
  outputs/tables/inference_per_cell.csv   50 cells x ~30 columns (HAC SE, p, CI)
  outputs/tables/cross_regime_z.csv       125 z-tests on coefficient changes
  outputs/tables/cross_regime_chow.csv    25 Chow F-tests for joint structural breaks
  outputs/tables/rolling_betas.csv        time series of rolling coefficients
  outputs/tables/rolling_se.csv           time series of HAC SEs
  outputs/tables/forecasts.csv            walk-forward OOS forecasts + PIs
  outputs/tables/oos_by_regime.csv        OOS metrics aggregated by regime
  outputs/tables/validity_per_cell.csv    composite validity score per cell
  outputs/tables/validity_per_regime.csv  validity averaged across sectors
  outputs/tables/validity_crisis_vs_calm.csv  crisis-vs-calm comparison
"""

(OUT / "00_methodology.txt").write_text(METHOD)
print("Methodology document written.")

# %% [markdown]
# ## Cell 5 -- STAGE 1: Build master panel
#
# Cleans T-bill data, restricts the index panel to 2011-2022, fixes the one
# bad Nifty IT data point (2016-04-15), computes weekly returns, excess
# returns, and assembles all four factors.

# %%
def stage1_build_panel():
    """Build master panel from the four uploaded CSVs."""
    # T-bill: auctions -> daily forward-fill -> weekly Friday yield
    df = pd.read_csv(INPUT / "tbill_91day_clean.csv")
    df["date_of_auction"] = pd.to_datetime(df["date_of_auction"])
    keep = (df[["date_of_auction", "weighted_avg_yield_pct_annual"]]
            .rename(columns={"date_of_auction": "date",
                             "weighted_avg_yield_pct_annual": "rf_annual_pct"})
            .sort_values("date").drop_duplicates("date"))

    daily_idx = pd.date_range("2004-12-01", "2023-01-31", freq="B")
    rf_daily  = keep.set_index("date").reindex(daily_idx, method="ffill").dropna()
    rf_daily.index.name = "date"
    rf_daily.to_csv(DATA / "rf_daily.csv")

    tbill_w = rf_daily["rf_annual_pct"].resample("W-FRI").last().dropna()
    tbill_w.to_csv(DATA / "tbill_91d_weekly.csv", header=["tbill_91d_pct"])
    rf_weekly = (1 + tbill_w / 100) ** (1 / 52) - 1
    rf_weekly.name = "rf_weekly"

    # Index panel: yfinance weekly file, Friday-anchored
    yf = pd.read_csv(INPUT / "nifty_weekly_2005_2022.csv",
                     parse_dates=["date"], index_col="date")
    yf.index = yf.index + pd.Timedelta(days=4)   # Mon-anchor -> Fri-anchor
    panel = yf.loc[WINDOW_START:WINDOW_END].copy()
    panel = panel.rename(columns={"NIFTYOILGAS": "NIFTYENERGY"})

    # Fix bad IT data point (yfinance returned 1060 vs ~11000)
    bad_date = pd.Timestamp("2016-04-15")
    if bad_date in panel.index and panel.loc[bad_date, "NIFTYIT"] < 5000:
        panel.loc[bad_date, "NIFTYIT"] = np.nan
        panel["NIFTYIT"] = panel["NIFTYIT"].interpolate(method="linear")
    panel.to_csv(DATA / "index_panel_2011_2022.csv")

    # Returns and excess returns
    returns = panel.pct_change().dropna(how="all")
    returns.columns = ["r_NIFTY50", "r_BANK", "r_IT", "r_PHARMA", "r_FMCG", "r_ENERGY"]
    returns.to_csv(DATA / "returns_weekly.csv")

    rf_aligned = rf_weekly.reindex(returns.index)
    rf_aligned.to_csv(DATA / "rf_weekly_aligned.csv", header=["rf_weekly"])

    excess = returns.subtract(rf_aligned, axis=0)
    excess.columns = ["xr_NIFTY50", "xr_BANK", "xr_IT", "xr_PHARMA", "xr_FMCG", "xr_ENERGY"]
    excess.to_csv(DATA / "excess_returns_weekly.csv")

    # G-Sec 10Y -> term spread
    gsec = pd.read_csv(INPUT / "India_10-Year_Bond_Yield_Historical_Data.csv")
    gsec.columns = [c.strip().replace("\ufeff", "") for c in gsec.columns]
    gsec["date"] = pd.to_datetime(gsec["Date"], format="%m/%d/%Y")
    gsec = (gsec[["date", "Price"]]
              .rename(columns={"Price": "gsec_10y_pct"})
              .sort_values("date").set_index("date"))
    gsec["gsec_10y_pct"] = pd.to_numeric(gsec["gsec_10y_pct"], errors="coerce")
    gsec_w = gsec["gsec_10y_pct"].resample("W-FRI").last().dropna()
    gsec_w.to_csv(DATA / "gsec_10y_weekly.csv", header=["gsec_10y_pct"])

    fri_grid = pd.date_range(WINDOW_START, WINDOW_END, freq="W-FRI")
    term_spread = (gsec_w.reindex(fri_grid) -
                   tbill_w.reindex(fri_grid)).rename("term_spread_pct")
    term_spread.to_csv(DATA / "term_spread_weekly.csv", header=["term_spread_pct"])

    # India VIX and USD/INR
    mf = pd.read_csv(INPUT / "macro_factors_weekly.csv",
                     parse_dates=["date"], index_col="date")
    mf = mf.loc[WINDOW_START:WINDOW_END]

    # Master panel
    mkt_excess    = excess["xr_NIFTY50"].rename("mkt_excess")
    d_term_spread = term_spread.diff().rename("d_term_spread")
    d_vix         = mf["INDIAVIX"].diff().rename("d_vix")
    r_inr         = mf["USDINR"].pct_change().rename("r_inr")

    master = pd.concat([excess, mkt_excess, d_term_spread, d_vix, r_inr],
                        axis=1, sort=True)
    master = master.dropna(subset=["mkt_excess", "d_term_spread", "d_vix", "r_inr"])
    master.index.name = "date"
    master.to_csv(DATA / "master_panel.csv")
    return master


print("=" * 80)
print("STAGE 1: BUILD MASTER PANEL")
print("=" * 80)
master = stage1_build_panel()
print(f"Master panel: {master.shape[0]} weeks x {master.shape[1]} columns")
print(f"Date range:   {master.index.min().date()} to {master.index.max().date()}")
print(f"Saved:        {DATA / 'master_panel.csv'}")

# %% [markdown]
# ## Cell 6 -- STAGE 2: Diagnostics (assumption verification A1-A6)
#
# For each (sector x regime) cell:
#
# - Ramsey RESET (linearity)
# - VIF on factors (multicollinearity, regime-level)
# - Breusch-Pagan, White, ARCH-LM (heteroskedasticity)
# - Durbin-Watson, Ljung-Box on residuals and squared residuals (autocorrelation)
# - Jarque-Bera (normality)
#
# Outputs: `outputs/tables/diag_table.csv`, `outputs/tables/vif_by_regime.csv`,
# and a readable summary at `outputs/diag_summary.txt`.

# %%
def fit_hac(y, X, lags=HAC_LAGS_FULL):
    Xc = sm.add_constant(X)
    return sm.OLS(y, Xc).fit(cov_type="HAC", cov_kwds={"maxlags": lags})


def stationarity_tests(s):
    s = s.dropna()
    if len(s) < 10:
        return {"adf_pval": np.nan, "kpss_pval": np.nan}
    adf_pval = adfuller(s, autolag="AIC")[1]
    try:
        kpss_pval = kpss(s, regression="c", nlags="auto")[1]
    except Exception:
        kpss_pval = np.nan
    return {"adf_pval": adf_pval, "kpss_pval": kpss_pval}


def diag_normality(resid):
    jb_stat, jb_pval, skew, kurt = jarque_bera(resid)
    return {"jb_pval": jb_pval, "skew": skew, "kurt": kurt,
            "normal": jb_pval > ALPHA}


def diag_hetero(resid, X):
    Xc = sm.add_constant(X)
    bp_pval = het_breuschpagan(resid, Xc)[1]
    try:
        white_pval = het_white(resid, Xc)[1]
    except Exception:
        white_pval = np.nan
    arch_pval = het_arch(resid, nlags=4)[1]
    return {"bp_pval": bp_pval, "white_pval": white_pval, "arch_pval": arch_pval,
            "homoskedastic": (bp_pval > ALPHA) and (arch_pval > ALPHA)}


def diag_autocorr(resid):
    dw  = durbin_watson(resid)
    lb  = acorr_ljungbox(resid,    lags=[4], return_df=True)["lb_pvalue"].iloc[0]
    lb2 = acorr_ljungbox(resid**2, lags=[4], return_df=True)["lb_pvalue"].iloc[0]
    return {"dw": dw, "lb_pval": lb, "lb_sq_pval": lb2,
            "no_autocorr": lb > ALPHA}


def diag_linearity(model_result):
    try:
        r = linear_reset(model_result, power=[2, 3], use_f=True)
        return {"reset_pval": r.pvalue, "linear": r.pvalue > ALPHA}
    except Exception:
        return {"reset_pval": np.nan, "linear": np.nan}


def vifs(X):
    Xc = sm.add_constant(X)
    return {col: variance_inflation_factor(Xc.values, i)
            for i, col in enumerate(Xc.columns) if col != "const"}


def diagnose_one(y, X):
    """Full battery for one (y, X) pair."""
    res = fit_hac(y, X)
    resid = res.resid
    out = {"n": int(res.nobs), "r2": res.rsquared, "r2_adj": res.rsquared_adj,
           "n_warn": int(res.nobs) < 30}
    for raw, nice in COEF_MAP.items():
        out[nice]              = res.params[raw]
        out[f"{nice}_se_hac"]  = res.bse[raw]
        out[f"{nice}_t_hac"]   = res.tvalues[raw]
        out[f"{nice}_p_hac"]   = res.pvalues[raw]
        lo, hi                 = res.conf_int().loc[raw]
        out[f"{nice}_ci_lo"]   = lo
        out[f"{nice}_ci_hi"]   = hi
    out.update(diag_linearity(res))
    out.update(diag_hetero(resid, X))
    out.update(diag_autocorr(resid))
    out.update(diag_normality(resid))
    return out


def stage2_diagnostics(master):
    rows, vif_rows = [], []
    full_sectors = ["xr_NIFTY50"] + SECTORS

    for regime_name, start, end, rtype in REGIMES:
        sub = master.loc[start:end]
        if sub.empty:
            continue
        Xreg = sub[FACTORS].dropna()
        v = vifs(Xreg)
        vif_rows.append({"regime": regime_name, "type": rtype,
                         "n": len(Xreg),
                         **{f"vif_{k}": val for k, val in v.items()}})
        for sector in full_sectors:
            df = sub[[sector] + FACTORS].dropna()
            if len(df) < 8:
                continue
            row = {"regime": regime_name, "type": rtype, "sector": sector}
            row.update(diagnose_one(df[sector], df[FACTORS]))
            rows.append(row)

    diag_df = pd.DataFrame(rows)
    vif_df  = pd.DataFrame(vif_rows)

    front = ["regime", "type", "sector", "n", "n_warn", "r2", "r2_adj"]
    other = [c for c in diag_df.columns if c not in front]
    diag_df = diag_df[front + other]

    diag_df.to_csv(TABLES / "diag_table.csv", index=False)
    vif_df.to_csv(TABLES / "vif_by_regime.csv", index=False)

    # Readable summary
    lines = []
    lines.append("=" * 100)
    lines.append("DIAGNOSTIC SUMMARY: 4-FACTOR MODEL  (regime-pooled, HAC SE, alpha=0.05)")
    lines.append("=" * 100)
    lines.append("Pass/fail flags (P=PASS at alpha=0.05):")
    lines.append("  linear        Ramsey RESET")
    lines.append("  homoskedastic Breusch-Pagan AND ARCH-LM")
    lines.append("  no_autocorr   Ljung-Box on residuals")
    lines.append("  normal        Jarque-Bera (failure expected for financial data)")
    lines.append("")
    for regime_name, start, end, rtype in REGIMES:
        block = diag_df[diag_df["regime"] == regime_name]
        if block.empty:
            continue
        lines.append(f"--- {regime_name} ({rtype}, {start} to {end}) ---")
        v = vif_df[vif_df["regime"] == regime_name].iloc[0]
        lines.append("  Factor VIF: " +
                     ", ".join(f"{c.replace('vif_',''):14s}={v[c]:.2f}"
                               for c in vif_df.columns if c.startswith("vif_")))
        lines.append(f"  {'Sector':12s} {'n':>4s} {'R2':>6s} "
                     f"{'lin':>4s} {'homo':>5s} {'noAC':>5s} {'norm':>5s} "
                     f"{'beta_M':>8s} {'beta_L':>10s} {'beta_FX':>9s} {'beta_TS':>9s}")
        for _, r in block.iterrows():
            f_ = lambda b: "?" if pd.isna(b) else ("P" if b else "F")
            lines.append(
                f"  {r['sector'].replace('xr_',''):12s} {int(r['n']):>4d} "
                f"{r['r2']:>6.3f} "
                f"{f_(r['linear']):>4s} {f_(r['homoskedastic']):>5s} "
                f"{f_(r['no_autocorr']):>5s} {f_(r['normal']):>5s} "
                f"{r['beta_M']:>+8.3f} {r['beta_L']:>+10.4f} "
                f"{r['beta_FX']:>+9.3f} {r['beta_TS']:>+9.3f}")
        if block["n_warn"].any():
            lines.append("  [!] n<30 in this regime; small-sample inference warning.")
        lines.append("")

    (OUT / "diag_summary.txt").write_text("\n".join(lines))
    return diag_df, vif_df


print()
print("=" * 80)
print("STAGE 2: DIAGNOSTICS  (verifying A1-A6)")
print("=" * 80)
diag_df, vif_df = stage2_diagnostics(master)
print(f"  Diagnostic cells: {len(diag_df)}")
print(f"  Saved: {TABLES / 'diag_table.csv'}")
print(f"  Saved: {TABLES / 'vif_by_regime.csv'}")
print(f"  Saved: {OUT / 'diag_summary.txt'}")

# %% [markdown]
# ## Cell 7 -- STAGE 3: Inference (t/F/z tests + 95% CIs)
#
# Per (sector x regime) cell, computes:
#
# - HAC-corrected t-statistics, p-values, 95% confidence intervals on every coefficient
# - test of $H_0: \beta_M = 1$ for each sector x regime
# - joint F-test that $\beta_L = \beta_{FX} = \beta_{TS} = 0$ (does multi-factor add over CAPM?)
#
# Per (regime pair x sector):
#
# - z-tests on each coefficient: $H_0: \beta_A = \beta_B$
# - Chow F-test on joint coefficient equality across regimes

# %%
def regime_slice(master, name):
    for nm, s, e, _ in REGIMES:
        if nm == name:
            return master.loc[s:e]
    raise KeyError(name)


def stage3_inference(master):
    rows = []
    for regime_name, start, end, rtype in REGIMES:
        sub = master.loc[start:end]
        for sector in SECTORS:
            df = sub[[sector] + FACTORS].dropna()
            if len(df) < 8:
                continue
            res = fit_hac(df[sector], df[FACTORS])
            n = int(res.nobs); dfres = n - len(FACTORS) - 1
            row = {"regime": regime_name, "type": rtype, "sector": sector,
                   "n": n, "r2": res.rsquared, "r2_adj": res.rsquared_adj,
                   "n_warn": n < 30}
            for raw, nice in COEF_MAP.items():
                row[nice]            = res.params[raw]
                row[f"{nice}_se"]    = res.bse[raw]
                row[f"{nice}_t"]     = res.tvalues[raw]
                row[f"{nice}_p"]     = res.pvalues[raw]
                lo, hi = res.conf_int().loc[raw]
                row[f"{nice}_ci_lo"] = lo
                row[f"{nice}_ci_hi"] = hi
            # H0: beta_M = 1
            b_mkt, se_mkt = res.params["mkt_excess"], res.bse["mkt_excess"]
            t_b1 = (b_mkt - 1) / se_mkt
            p_b1 = 2 * (1 - stats.t.cdf(abs(t_b1), df=dfres))
            row["t_beta_M_eq_1"] = t_b1
            row["p_beta_M_eq_1"] = p_b1
            # Joint F: do non-market factors add over CAPM?
            ftest = res.f_test("d_vix = 0, r_inr = 0, d_term_spread = 0")
            row["F_nonmkt_joint"]   = float(ftest.fvalue)
            row["p_F_nonmkt_joint"] = float(ftest.pvalue)
            rows.append(row)

    per_cell = pd.DataFrame(rows)
    front = ["regime","type","sector","n","n_warn","r2","r2_adj"]
    per_cell = per_cell[front + [c for c in per_cell.columns if c not in front]]
    per_cell.to_csv(TABLES / "inference_per_cell.csv", index=False)

    # Cross-regime z-tests + Chow F
    rows_z, rows_F = [], []
    for sector in SECTORS:
        for rA, rB in REGIME_PAIRS:
            dfA = regime_slice(master, rA)[[sector] + FACTORS].dropna()
            dfB = regime_slice(master, rB)[[sector] + FACTORS].dropna()
            if len(dfA) < 8 or len(dfB) < 8:
                continue
            resA = fit_hac(dfA[sector], dfA[FACTORS])
            resB = fit_hac(dfB[sector], dfB[FACTORS])
            for raw, nice in COEF_MAP.items():
                bA, seA = resA.params[raw], resA.bse[raw]
                bB, seB = resB.params[raw], resB.bse[raw]
                z = (bA - bB) / np.sqrt(seA**2 + seB**2)
                p = 2 * (1 - stats.norm.cdf(abs(z)))
                rows_z.append({"sector": sector, "regime_A": rA, "regime_B": rB,
                               "coef": nice,
                               "beta_A": bA, "se_A": seA, "beta_B": bB, "se_B": seB,
                               "diff": bA - bB, "z": z, "p": p,
                               "reject_5pct": p < ALPHA})
            # Chow F
            pool = pd.concat([dfA.assign(D=0), dfB.assign(D=1)], axis=0).copy()
            for f in FACTORS:
                pool[f"D_x_{f}"] = pool["D"] * pool[f]
            X_cols = ["D"] + FACTORS + [f"D_x_{f}" for f in FACTORS]
            res_pool = fit_hac(pool[sector], pool[X_cols])
            hyp = ["D = 0"] + [f"D_x_{f} = 0" for f in FACTORS]
            ftest = res_pool.f_test(", ".join(hyp))
            rows_F.append({"sector": sector, "regime_A": rA, "regime_B": rB,
                           "n_A": int(resA.nobs), "n_B": int(resB.nobs),
                           "F_chow": float(ftest.fvalue),
                           "p_F_chow": float(ftest.pvalue),
                           "reject_5pct": float(ftest.pvalue) < ALPHA})

    z_df = pd.DataFrame(rows_z)
    F_df = pd.DataFrame(rows_F)
    z_df.to_csv(TABLES / "cross_regime_z.csv",    index=False)
    F_df.to_csv(TABLES / "cross_regime_chow.csv", index=False)

    # Readable summary
    lines = []
    lines.append("=" * 100)
    lines.append("INFERENCE SUMMARY: 4-FACTOR MODEL  (HAC SE, alpha=0.05)")
    lines.append("=" * 100)
    lines.append("\nPER-REGIME COEFFICIENT ESTIMATES (* = p<0.05)")
    lines.append("-" * 100)

    def s_(b, p):
        star = "*" if (not pd.isna(p) and p < ALPHA) else " "
        return f"{b:+.3f}{star}"

    for regime_name, _, _, rtype in REGIMES:
        block = per_cell[per_cell["regime"] == regime_name]
        if block.empty:
            continue
        lines.append(f"\n[{regime_name}] ({rtype}, n={int(block['n'].iloc[0])})")
        lines.append(f"  {'Sector':10s}  {'beta_M':>10s} {'beta_L':>11s} "
                     f"{'beta_FX':>10s} {'beta_TS':>10s}    {'beta_M=1':>10s} {'F_joint':>10s}")
        for _, r in block.iterrows():
            sb1 = "*" if r["p_beta_M_eq_1"]   < ALPHA else " "
            sFj = "*" if r["p_F_nonmkt_joint"] < ALPHA else " "
            lines.append(
                f"  {r['sector'].replace('xr_',''):10s}  "
                f"{s_(r['beta_M'],   r['beta_M_p']):>10s} "
                f"{s_(r['beta_L'],   r['beta_L_p']):>11s} "
                f"{s_(r['beta_FX'],  r['beta_FX_p']):>10s} "
                f"{s_(r['beta_TS'],  r['beta_TS_p']):>10s}    "
                f"p={r['p_beta_M_eq_1']:.3f}{sb1} "
                f"p={r['p_F_nonmkt_joint']:.3f}{sFj}")

    lines.append("")
    lines.append("=" * 100)
    lines.append("CHOW F-TEST: are coefficients identical across regime pairs?")
    lines.append("-" * 100)
    lines.append(f"  {'Sector':10s} {'Regime A':18s} {'Regime B':18s} {'F':>8s} "
                 f"{'p-value':>10s}  reject5%")
    for _, r in F_df.iterrows():
        flag = "***" if r["reject_5pct"] else "   "
        lines.append(f"  {r['sector'].replace('xr_',''):10s} "
                     f"{r['regime_A']:18s} {r['regime_B']:18s} "
                     f"{r['F_chow']:>8.3f} {r['p_F_chow']:>10.4f}  {flag}")

    lines.append("")
    lines.append("=" * 100)
    lines.append("Z-TESTS: change in MARKET BETA across regime pairs")
    lines.append("-" * 100)
    z_M = z_df[z_df["coef"] == "beta_M"]
    lines.append(f"  {'Sector':10s} {'Regime A':18s} {'Regime B':18s}  "
                 f"{'beta_A':>8s} {'beta_B':>8s} {'diff':>8s} {'z':>7s} {'p':>8s}")
    for _, r in z_M.iterrows():
        flag = "*" if r["reject_5pct"] else " "
        lines.append(f"  {r['sector'].replace('xr_',''):10s} "
                     f"{r['regime_A']:18s} {r['regime_B']:18s}  "
                     f"{r['beta_A']:>+8.3f} {r['beta_B']:>+8.3f} "
                     f"{r['diff']:>+8.3f} {r['z']:>+7.2f} {r['p']:>7.4f}{flag}")

    (OUT / "inference_summary.txt").write_text("\n".join(lines))
    return per_cell, z_df, F_df


print()
print("=" * 80)
print("STAGE 3: INFERENCE  (t/F/z tests + 95% CIs)")
print("=" * 80)
per_cell, z_df, F_df = stage3_inference(master)
print(f"  Per-cell inference: {len(per_cell)} rows")
print(f"  z-tests:            {len(z_df)} rows")
print(f"  Chow F-tests:       {len(F_df)} rows")
print(f"  Saved: {TABLES / 'inference_per_cell.csv'}")
print(f"  Saved: {TABLES / 'cross_regime_z.csv'}")
print(f"  Saved: {TABLES / 'cross_regime_chow.csv'}")
print(f"  Saved: {OUT / 'inference_summary.txt'}")

# %% [markdown]
# ## Cell 8 -- STAGE 4: Rolling OLS estimation + walk-forward OOS forecasting
#
# 52-week rolling window, 1-week step, HAC SEs with maxlags=3.
# Each window forecasts the next observation; prediction errors and 95% PIs
# are recorded.  OOS metrics are aggregated by (sector x regime).

# %%
def stage4_rolling(master):
    rows_b, rows_se, rows_f = [], [], []
    for sector in SECTORS:
        df = master[[sector] + FACTORS].dropna().copy()
        n = len(df)
        if n < ROLLING_WINDOW + 1:
            continue
        for t_end in range(ROLLING_WINDOW - 1, n - 1):
            win = df.iloc[t_end - ROLLING_WINDOW + 1: t_end + 1]
            try:
                res = fit_hac(win[sector], win[FACTORS], lags=HAC_LAGS_ROLL)
            except Exception:
                continue
            d_end = win.index[-1]
            rows_b.append({"date": d_end, "sector": sector,
                           "alpha":   res.params["const"],
                           "beta_M":  res.params["mkt_excess"],
                           "beta_L":  res.params["d_vix"],
                           "beta_FX": res.params["r_inr"],
                           "beta_TS": res.params["d_term_spread"],
                           "r2": res.rsquared, "n": int(res.nobs)})
            rows_se.append({"date": d_end, "sector": sector,
                            "alpha_se":   res.bse["const"],
                            "beta_M_se":  res.bse["mkt_excess"],
                            "beta_L_se":  res.bse["d_vix"],
                            "beta_FX_se": res.bse["r_inr"],
                            "beta_TS_se": res.bse["d_term_spread"]})
            # walk-forward forecast for t+1
            nxt = df.iloc[t_end + 1]
            x_next = np.array([1.0, nxt["mkt_excess"], nxt["d_vix"],
                               nxt["r_inr"], nxt["d_term_spread"]])
            y_pred = float(x_next @ res.params.values)
            y_true = float(nxt[sector])
            resid_var = float(np.var(res.resid, ddof=len(res.params)))
            beta_var  = res.cov_params().values
            pred_var  = resid_var + float(x_next @ beta_var @ x_next)
            pi_lo, pi_hi = y_pred - 1.96*np.sqrt(pred_var), y_pred + 1.96*np.sqrt(pred_var)
            d_fcst = df.index[t_end + 1]
            rows_f.append({"date": d_fcst, "sector": sector,
                           "y_true": y_true, "y_pred": y_pred,
                           "error": y_true - y_pred,
                           "sq_error": (y_true - y_pred)**2,
                           "pi_lo": pi_lo, "pi_hi": pi_hi,
                           "in_pi95": (pi_lo <= y_true <= pi_hi),
                           "regime": assign_regime(d_fcst)})

    betas_long = pd.DataFrame(rows_b)
    se_long    = pd.DataFrame(rows_se)
    fcst_df    = pd.DataFrame(rows_f)
    betas_long.to_csv(TABLES / "rolling_betas.csv", index=False)
    se_long.to_csv(   TABLES / "rolling_se.csv",    index=False)
    fcst_df.to_csv(   TABLES / "forecasts.csv",     index=False)

    # OOS aggregate
    rows = []
    for sector in SECTORS:
        for regime_name, _, _, rtype in REGIMES:
            sub = fcst_df[(fcst_df["sector"] == sector) &
                          (fcst_df["regime"] == regime_name)]
            if sub.empty:
                continue
            mse = sub["sq_error"].mean()
            ss_res = (sub["error"]**2).sum()
            ss_tot_zero = (sub["y_true"]**2).sum()
            oos_r2_zero = 1.0 - ss_res / ss_tot_zero if ss_tot_zero > 0 else np.nan
            mean_y = sub["y_true"].mean()
            ss_tot_mean = ((sub["y_true"] - mean_y)**2).sum()
            oos_r2_mean = 1.0 - ss_res / ss_tot_mean if ss_tot_mean > 0 else np.nan
            rows.append({"sector": sector, "regime": regime_name, "type": rtype,
                         "n_pred": len(sub),
                         "mse": mse, "rmse": np.sqrt(mse),
                         "mae": sub["error"].abs().mean(),
                         "oos_r2_zero": oos_r2_zero,
                         "oos_r2_histmean": oos_r2_mean,
                         "pi_coverage_95": sub["in_pi95"].mean()})
    oos_df = pd.DataFrame(rows)
    oos_df.to_csv(TABLES / "oos_by_regime.csv", index=False)

    # Readable summary
    lines = []
    lines.append("=" * 100)
    lines.append("ROLLING 4-FACTOR MODEL  (52-week window, 1-week step, HAC SE lag=3)")
    lines.append("=" * 100)
    lines.append("\nROLLING-BETA SUMMARY (mean, std across all windows)")
    lines.append(f"  {'Sector':10s} {'mean(beta_M)':>14s} {'std(beta_M)':>14s} "
                 f"{'min':>8s} {'max':>8s}  {'n_windows':>10s}")
    for sector in SECTORS:
        sub = betas_long[betas_long["sector"] == sector]
        if sub.empty:
            continue
        b = sub["beta_M"]
        lines.append(f"  {sector.replace('xr_',''):10s} "
                     f"{b.mean():>14.3f} {b.std():>14.3f} "
                     f"{b.min():>8.3f} {b.max():>8.3f}  {len(sub):>10d}")

    lines.append("\nMARKET-BETA STABILITY BY REGIME  (std of rolling beta_M within each regime)")
    headers = "  " + "Sector".ljust(10) + " " + " ".join(f"{r[0][:14]:>14s}" for r in REGIMES)
    lines.append(headers)
    for sector in SECTORS:
        cells = [sector.replace("xr_","").ljust(10)]
        for regime_name, start, end, _ in REGIMES:
            sub = betas_long[(betas_long["sector"] == sector) &
                             (betas_long["date"] >= start) & (betas_long["date"] <= end)]
            if sub.empty or len(sub) < 2:
                cells.append(f"{'-':>14s}")
            else:
                cells.append(f"{sub['beta_M'].std():>14.3f}")
        lines.append("  " + " ".join(cells))

    lines.append("\n" + "=" * 100)
    lines.append("OOS FORECAST METRICS BY (SECTOR x REGIME)")
    lines.append("=" * 100)
    for sector in SECTORS:
        sub = oos_df[oos_df["sector"] == sector]
        if sub.empty:
            continue
        lines.append(f"\n[{sector.replace('xr_','')}]")
        lines.append(f"  {'Regime':18s} {'type':8s} {'n':>4s} {'RMSE':>8s} "
                     f"{'OOS_R2_0':>10s} {'OOS_R2_mu':>10s} {'PI_cov':>8s}")
        for _, r in sub.iterrows():
            lines.append(f"  {r['regime']:18s} {r['type']:8s} {int(r['n_pred']):>4d} "
                         f"{r['rmse']:>8.4f} {r['oos_r2_zero']:>+10.3f} "
                         f"{r['oos_r2_histmean']:>+10.3f} {r['pi_coverage_95']:>8.2%}")

    lines.append("\n" + "=" * 100)
    lines.append("CRISIS vs CALM: average OOS performance pooled across sectors")
    lines.append("-" * 100)
    crisis = oos_df[oos_df["type"] == "crisis"]
    calm   = oos_df[oos_df["type"] == "calm"]
    lines.append(f"  {'group':10s} {'avg_RMSE':>10s} {'avg_OOS_R2_0':>14s} "
                 f"{'avg_OOS_R2_mu':>14s} {'avg_PI_cov':>10s}")
    for label, grp in [("crisis", crisis), ("calm", calm)]:
        lines.append(f"  {label:10s} {grp['rmse'].mean():>10.4f} "
                     f"{grp['oos_r2_zero'].mean():>+14.3f} "
                     f"{grp['oos_r2_histmean'].mean():>+14.3f} "
                     f"{grp['pi_coverage_95'].mean():>10.2%}")

    (OUT / "rolling_summary.txt").write_text("\n".join(lines))
    return betas_long, se_long, fcst_df, oos_df


print()
print("=" * 80)
print("STAGE 4: ROLLING ESTIMATION + WALK-FORWARD OOS")
print("=" * 80)
betas_long, se_long, fcst_df, oos_df = stage4_rolling(master)
print(f"  Rolling betas: {len(betas_long)} rows")
print(f"  Forecasts:     {len(fcst_df)} rows")
print(f"  OOS by regime: {len(oos_df)} rows")
print(f"  Saved: {TABLES / 'rolling_betas.csv'}")
print(f"  Saved: {TABLES / 'rolling_se.csv'}")
print(f"  Saved: {TABLES / 'forecasts.csv'}")
print(f"  Saved: {TABLES / 'oos_by_regime.csv'}")
print(f"  Saved: {OUT / 'rolling_summary.txt'}")

# %% [markdown]
# ## Cell 9 -- STAGE 5: Regime-wise validity assessment
#
# Composite score (in [0, 1]) per (sector x regime), aggregated to a
# per-regime verdict and a crisis-vs-calm comparison.

# %%
def stage5_validity(diag_df, per_cell, oos_df, betas_long):
    clip01 = lambda s: s.clip(lower=0.0, upper=1.0)
    diag = diag_df[diag_df["sector"].isin(SECTORS)].copy()

    c1 = per_cell[["regime","sector","r2","n"]].rename(columns={"r2": "r2_is"})
    c2 = oos_df[["regime","sector","oos_r2_zero","pi_coverage_95","n_pred"]].copy()
    c2["oos_r2_score"] = clip01(c2["oos_r2_zero"])
    c2["pi_score"]     = clip01(1.0 - (c2["pi_coverage_95"] - 0.95).abs())
    diag["diag_pass"] = (
        diag["linear"].fillna(False).astype(int)
        + diag["homoskedastic"].fillna(False).astype(int)
        + diag["no_autocorr"].fillna(False).astype(int)
    ) / 3.0
    c3 = diag[["regime","sector","diag_pass"]]

    stab = []
    for sector in SECTORS:
        for regime_name, start, end, _ in REGIMES:
            sub = betas_long[(betas_long["sector"] == sector) &
                             (betas_long["date"] >= pd.Timestamp(start)) &
                             (betas_long["date"] <= pd.Timestamp(end))]
            if len(sub) < 2:
                stab.append({"sector": sector, "regime": regime_name,
                             "beta_M_std": np.nan, "beta_stab": np.nan})
            else:
                std_b = sub["beta_M"].std()
                stab.append({"sector": sector, "regime": regime_name,
                             "beta_M_std": std_b,
                             "beta_stab": 1.0 / (1.0 + std_b)})
    c4 = pd.DataFrame(stab)

    df = (c1.merge(c2, on=["regime","sector"], how="outer")
            .merge(c3, on=["regime","sector"], how="outer")
            .merge(c4, on=["regime","sector"], how="outer"))
    df["r2_is_score"] = clip01(df["r2_is"])
    components = ["r2_is_score","oos_r2_score","diag_pass","pi_score","beta_stab"]
    df["validity_score"] = df[components].mean(axis=1, skipna=True)

    type_lookup = {nm: t for nm, _, _, t in REGIMES}
    df["type"] = df["regime"].map(type_lookup)

    front = ["regime","type","sector","n","n_pred",
             "r2_is","oos_r2_zero","pi_coverage_95","beta_M_std",
             "r2_is_score","oos_r2_score","diag_pass","pi_score","beta_stab",
             "validity_score"]
    df = df[[c for c in front if c in df.columns]]
    df.to_csv(TABLES / "validity_per_cell.csv", index=False)

    by_regime = df.groupby(["regime","type"], sort=False).agg(
        n_avg=("n","mean"),
        in_sample_r2=("r2_is","mean"),
        oos_r2=("oos_r2_zero","mean"),
        pi_coverage=("pi_coverage_95","mean"),
        beta_stability=("beta_stab","mean"),
        diag_pass_rate=("diag_pass","mean"),
        validity_score=("validity_score","mean")
    ).reset_index()
    by_regime.to_csv(TABLES / "validity_per_regime.csv", index=False)

    by_type = df.groupby("type").agg(
        in_sample_r2=("r2_is","mean"),
        oos_r2=("oos_r2_zero","mean"),
        pi_coverage=("pi_coverage_95","mean"),
        beta_stability=("beta_stab","mean"),
        diag_pass_rate=("diag_pass","mean"),
        validity_score=("validity_score","mean")
    ).reset_index()
    by_type.to_csv(TABLES / "validity_crisis_vs_calm.csv", index=False)

    # Readable summary
    lines = []
    lines.append("=" * 100)
    lines.append("MODEL VALIDITY ASSESSMENT  (4-factor model, regime-wise)")
    lines.append("=" * 100)
    lines.append("\nValidity score = mean of: in-sample R^2, OOS R^2 (clipped),")
    lines.append("                 diagnostic pass rate (linearity / homoskedasticity / no-autocorr),")
    lines.append("                 PI coverage closeness to 95%, beta stability.")
    lines.append("All components in [0, 1]; higher = more reliable.")
    lines.append("\nVALIDITY BY REGIME (averaged across 5 sectors):")
    lines.append(f"  {'Regime':18s} {'type':8s} {'in-R2':>7s} {'OOS-R2':>8s} "
                 f"{'PI cov':>8s} {'b-stab':>7s} {'diag':>6s}  {'SCORE':>7s}")
    for _, r in by_regime.iterrows():
        lines.append(f"  {r['regime']:18s} {r['type']:8s} "
                     f"{r['in_sample_r2']:>7.3f} {r['oos_r2']:>+8.3f} "
                     f"{r['pi_coverage']:>8.2%} {r['beta_stability']:>7.3f} "
                     f"{r['diag_pass_rate']:>6.2f}  {r['validity_score']:>7.3f}")

    lines.append("\n" + "=" * 100)
    lines.append("HEADLINE: CRISIS vs CALM REGIMES")
    lines.append("-" * 100)
    lines.append(f"  {'Type':10s} {'in-R2':>7s} {'OOS-R2':>8s} "
                 f"{'PI cov':>8s} {'b-stab':>7s} {'diag':>6s}  {'SCORE':>7s}")
    for _, r in by_type.iterrows():
        lines.append(f"  {r['type']:10s} {r['in_sample_r2']:>7.3f} "
                     f"{r['oos_r2']:>+8.3f} {r['pi_coverage']:>8.2%} "
                     f"{r['beta_stability']:>7.3f} {r['diag_pass_rate']:>6.2f}  "
                     f"{r['validity_score']:>7.3f}")
    if "crisis" in by_type["type"].values and "calm" in by_type["type"].values:
        c = by_type.set_index("type")
        delta = c.loc["calm"] - c.loc["crisis"]
        lines.append("\n  DELTA (calm - crisis); positive => calm is more reliable:")
        for col, lbl in [("in_sample_r2","in-sample R^2"),
                         ("oos_r2","OOS R^2"),
                         ("pi_coverage","PI coverage"),
                         ("beta_stability","beta stability"),
                         ("diag_pass_rate","diagnostic pass rate"),
                         ("validity_score","VALIDITY SCORE")]:
            lines.append(f"    {lbl:25s}: {delta[col]:+.3f}")

    lines.append("\n" + "=" * 100)
    lines.append("FIVE LEAST-RELIABLE (sector x regime) CELLS")
    lines.append("-" * 100)
    worst = df.nsmallest(5, "validity_score")
    lines.append(f"  {'Regime':18s} {'Sector':10s} {'in-R2':>7s} {'OOS-R2':>8s} "
                 f"{'PI cov':>8s} {'SCORE':>7s}")
    for _, r in worst.iterrows():
        lines.append(f"  {r['regime']:18s} {r['sector'].replace('xr_',''):10s} "
                     f"{r['r2_is']:>7.3f} {r['oos_r2_zero']:>+8.3f} "
                     f"{r['pi_coverage_95']:>8.2%} {r['validity_score']:>7.3f}")

    lines.append("\nFIVE MOST-RELIABLE (sector x regime) CELLS")
    lines.append("-" * 100)
    best = df.nlargest(5, "validity_score")
    lines.append(f"  {'Regime':18s} {'Sector':10s} {'in-R2':>7s} {'OOS-R2':>8s} "
                 f"{'PI cov':>8s} {'SCORE':>7s}")
    for _, r in best.iterrows():
        lines.append(f"  {r['regime']:18s} {r['sector'].replace('xr_',''):10s} "
                     f"{r['r2_is']:>7.3f} {r['oos_r2_zero']:>+8.3f} "
                     f"{r['pi_coverage_95']:>8.2%} {r['validity_score']:>7.3f}")

    (OUT / "validity_summary.txt").write_text("\n".join(lines))
    return df, by_regime, by_type


print()
print("=" * 80)
print("STAGE 5: REGIME-WISE VALIDITY")
print("=" * 80)
val_df, val_by_regime, val_by_type = stage5_validity(diag_df, per_cell, oos_df, betas_long)
print(f"  Per-cell validity:    {len(val_df)} rows")
print(f"  Per-regime aggregate: {len(val_by_regime)} rows")
print(f"  Crisis-vs-calm:       {len(val_by_type)} rows")
print(f"  Saved: {TABLES / 'validity_per_cell.csv'}")
print(f"  Saved: {TABLES / 'validity_per_regime.csv'}")
print(f"  Saved: {TABLES / 'validity_crisis_vs_calm.csv'}")
print(f"  Saved: {OUT / 'validity_summary.txt'}")

# %% [markdown]
# ## Cell 10 -- Print the validity summary (final headline)

# %%
print((OUT / "validity_summary.txt").read_text())

# %% [markdown]
# ## Cell 11 -- Bundle all outputs into a zip file and download

# %%
zip_path = CWD / "results_4factor.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(CWD))
print(f"Bundled all outputs: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024:.1f} KB")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(f"Not in Colab. Find your results at: {zip_path}")

Input directory : /content
Data directory  : /content/data
Outputs root    : /content/outputs
Tables folder   : /content/outputs/tables
Colab detected. Use the file picker below to select the 4 CSVs.


Saving tbill_91day_clean.csv to tbill_91day_clean (1).csv
Saving nifty_weekly_2005_2022.csv to nifty_weekly_2005_2022 (1).csv
Saving India_10-Year_Bond_Yield_Historical_Data.csv to India_10-Year_Bond_Yield_Historical_Data.csv
Saving macro_factors_weekly.csv to macro_factors_weekly (1).csv
  uploaded: tbill_91day_clean (1).csv
  uploaded: nifty_weekly_2005_2022 (1).csv
  uploaded: India_10-Year_Bond_Yield_Historical_Data.csv
  uploaded: macro_factors_weekly (1).csv
Methodology document written.
STAGE 1: BUILD MASTER PANEL
Master panel: 624 weeks x 10 columns
Date range:   2011-01-21 to 2022-12-30
Saved:        /content/data/master_panel.csv

STAGE 2: DIAGNOSTICS  (verifying A1-A6)
  Diagnostic cells: 60
  Saved: /content/outputs/tables/diag_table.csv
  Saved: /content/outputs/tables/vif_by_regime.csv
  Saved: /content/outputs/diag_summary.txt

STAGE 3: INFERENCE  (t/F/z tests + 95% CIs)
  Per-cell inference: 50 rows
  z-tests:            125 rows
  Chow F-tests:       25 rows
  Saved: /

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>